# Demo: flujo de delegación del orquestador

Este notebook demuestra el flujo típico:

`supervisor → research_agent → supervisor → analyst_agent → supervisor → END`

**Requisitos:** `.env` con `LLM_PROVIDER`, la API key del proveedor y `TAVILY_API_KEY`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print("LLM_PROVIDER:", os.getenv("LLM_PROVIDER"))
print("TAVILY_API_KEY set:", bool(os.getenv("TAVILY_API_KEY")))

## 1) Diagrama Mermaid del grafo

In [ ]:
from graph import get_graph_mermaid

print(get_graph_mermaid())

## 2) Ejecutar con streaming (ver delegación nodo a nodo)

In [ ]:
from langchain_core.messages import HumanMessage
from graph import graph

query = (
    "Que es LangGraph y que tan util es para orquestar multiagentes? "
    "Respuesta breve."
)

initial_state = {
    "messages": [HumanMessage(content=query)],
    "user_query": query,
    "next_agent": "research_agent",
    "research_findings": "",
    "analysis_result": "",
    "final_response": "",
    "task_completed": False,
    "step_count": 0,
    "validation_passed": False,
    "validation_feedback": "",
}

print("Consulta:", query)
print("\n--- Stream de actualizaciones ---\n")

final_state = None
for mode, chunk in graph.stream(
    initial_state,
    stream_mode=["updates", "values"],
):
    if mode == "updates":
        for node_name, payload in chunk.items():
            print(f"[{node_name}]")
            if isinstance(payload, dict):
                if payload.get("next_agent"):
                    print("  next_agent:", payload.get("next_agent"))
                if payload.get("research_findings"):
                    preview = str(payload["research_findings"])[:180].replace("\n", " ")
                    print("  research_findings:", preview, "...")
                if payload.get("analysis_result"):
                    preview = str(payload["analysis_result"])[:180].replace("\n", " ")
                    print("  analysis_result:", preview, "...")
                if payload.get("final_response"):
                    preview = str(payload["final_response"])[:180].replace("\n", " ")
                    print("  final_response:", preview, "...")
                if "step_count" in payload:
                    print("  step_count:", payload.get("step_count"))
            print()
    elif mode == "values":
        final_state = chunk

## 3) Resultado final

In [ ]:
print("=== Respuesta final ===\n")
print(final_state.get("final_response") or final_state.get("analysis_result"))
print("\n=== Metadatos ===")
print("steps:", final_state.get("step_count"))
print("task_completed:", final_state.get("task_completed"))
print("validation_passed:", final_state.get("validation_passed"))
print("next_agent:", final_state.get("next_agent"))

agent_names = [
    getattr(message, "name", None)
    for message in (final_state.get("messages") or [])
    if getattr(message, "name", None)
]
print("secuencia de agentes:", agent_names)